# Two AI agents on your own laptop

The same writer and reviewer as in `ai-agents.ipynb` (Module 9 Part 1), with one difference: the language model runs **on your own computer**, using a free program called **Ollama**.

- No API key.
- No cost.
- Nothing you type leaves your laptop.

**Before you start**, follow the setup sheet `resources/ollama-setup.md`. In short:

1. Install Ollama from https://ollama.com/download
2. In a terminal: `ollama pull llama3.2`
3. In your `dsai` environment: `pip install ollama`

In [1]:
import ollama

MODEL = "llama3.2"   # the model you downloaded with "ollama pull"

## Is Ollama ready?

This cell asks Ollama which models are on your computer, and looks for ours.

If it fails with `ConnectionError: Failed to connect to Ollama`, the Ollama app is not running. Start it, then run this cell again.

In [2]:
installed = ollama.list()    # every model on this computer

found = False
for model in installed.models:
    print("Installed:", model.model)
    if model.model.startswith(MODEL):
        found = True

if found:
    print("OK:", MODEL, "is ready")
else:
    print("Not found. In a terminal, run:  ollama pull", MODEL)

Installed: llama3.2:latest
Installed: nomic-embed-text:latest
OK: llama3.2 is ready


## One function to talk to the model

Compare this with `ask` in `ai-agents.ipynb`. The idea is the same: a **role** (who the model should be) and a **message** (what we ask).

One small difference: here the role goes **inside** the messages list, as a message from `"system"`. With Claude it had its own `system=` setting.

In [3]:
def ask(role, message):
    # the conversation: the role first, then our question
    conversation = [
        {"role": "system", "content": role},
        {"role": "user", "content": message},
    ]

    # send it to the model running on this computer
    response = ollama.chat(model=MODEL, messages=conversation)

    # keep only the text of the reply
    return response.message.content

## A first question

Before running: how long will the reply take? The first question is usually the slowest, because Ollama has to load the model into memory first.

In [4]:
import time

start = time.time()
reply = ask("Reply in one short sentence.", "What is a neural network?")
seconds = time.time() - start

print(reply)
print()
print("Took", round(seconds, 1), "seconds")

A neural network is a computer system inspired by the human brain, composed of interconnected nodes (neurons) that process and transmit information.

Took 3.7 seconds


## The team: a writer and a reviewer

The roles and the task are copied word for word from `ai-agents.ipynb`.

In [5]:
WRITER = "You are a friendly teacher. Explain ideas simply for beginners. Reply with the explanation only."
REVIEWER = "You review teaching material. Give at most three short bullet points: what is wrong, unclear or too hard."

TASK = "Explain Q-learning to a beginner in exactly three sentences."

Before running: this model is much smaller than Claude. Will the writer keep to exactly three sentences? Will the reviewer notice if it does not?

In [6]:
# 1. the writer drafts
draft = ask(WRITER, TASK)
print("DRAFT:\n", draft, "\n")

# 2. the reviewer gives feedback (and knows the three-sentence limit)
feedback = ask(REVIEWER, "The writer was asked: " + TASK + "\n\nReview this:\n" + draft)
print("FEEDBACK:\n", feedback, "\n")

# 3. the writer improves the draft
final = ask(WRITER, TASK + "\n\nYour draft:\n" + draft + "\n\nFeedback:\n" + feedback + "\n\nWrite a better version.")
print("FINAL:\n", final)

DRAFT:
 Q-learning is a type of machine learning algorithm that helps computers make decisions by learning from their experiences and rewards or penalties they receive for those actions. The computer learns a mathematical model, called a Q-table, which maps each state (situation) to an action (choice), with the value representing how good or bad the chosen action is in that situation. By updating this Q-table over time based on the rewards received, the algorithm can improve its decision-making skills and eventually make informed choices in new situations. 



FEEDBACK:
 Here are three points for improvement:

* The third sentence could be more concise and focused on the key aspect of Q-learning: improving decision-making through experience.
* The use of phrases like "how good or bad" is a bit vague; consider using more specific language, such as "maximum expected reward".
* Consider adding more context or examples to help beginners understand the practical application of Q-learning. 



FINAL:
 Here's an improved explanation:

Q-learning is a type of machine learning algorithm that helps computers make decisions by learning from their experiences and rewards or penalties they receive for those actions. The computer learns a mathematical model, called a Q-table, which maps each state (situation) to an action (choice), with the value representing the maximum expected reward for choosing that action in that situation. By updating its Q-table based on rewards, Q-learning enables computers to improve their decision-making over time and make informed choices in new situations.


## Claude or a local model?

| | Claude (`ai-agents.ipynb`) | Ollama with llama3.2 (this notebook) |
|---|---|---|
| Where it runs | Anthropic's computers | your laptop |
| API key | needed | not needed |
| Cost | a small charge per question | free |
| Your text | sent over the internet | stays on your laptop |
| Model size | very large | small (about 3 billion numbers) |
| Answer quality | strong | weaker: may ignore rules such as "exactly three sentences" |

**Takeaways**

- The three steps (draft, review, improve) did not change at all. Only the inside of `ask` changed. An agent pattern does not belong to one company's model.
- Each agent is still the same model with a different role.
- A small local model is free and private, but makes more mistakes. The reviewer can be wrong too, so a person still checks.
- Run the cell again: the replies change every time.